In [ ]:
import os
import gzip
import json
from typing import Any, Dict, List, Optional, Sequence, Set

def _load_saved_layer_contexts(
    *,
    saved_dir: str,
    layer_index: int,
    top_k: int,
) -> Dict[str, List[Dict[str, Any]]]:
    """Load saved contexts for a layer from JSON/JSON.GZ.

    Tries the following filenames in order:
      - L{layer}.top{top_k}.json.gz
      - L{layer}.top{top_k}.json
      - L{layer}.json.gz
      - L{layer}.json
    Returns a mapping: latent_id (str) -> list[record].
    """
    candidates = [
        os.path.join(saved_dir, f"L{layer_index}.top{top_k}.json.gz"),
        os.path.join(saved_dir, f"L{layer_index}.top{top_k}.json"),
        os.path.join(saved_dir, f"L{layer_index}.json.gz"),
        os.path.join(saved_dir, f"L{layer_index}.json"),
    ]
    path: Optional[str] = None
    for cand in candidates:
        if os.path.isfile(cand):
            path = cand
            break
    if path is None:
        raise FileNotFoundError(
            f"No saved contexts file found for layer {layer_index} under {saved_dir}"
        )

    opener = gzip.open if path.endswith(".gz") else open
    with opener(path, "rt") as f:
        data = json.load(f)
        if not isinstance(data, dict):
            raise ValueError(f"Saved contexts file has unexpected format: {path}")

    return data

In [ ]:
def extract_latent_window_tokens(
    layer_map: Dict[str, Any],
    window_size: int = 5,
    context_size: Optional[int] = 20
) -> Dict[int, List[List[Any]]]:
    """Extract tokens in windows around max-activating positions for each latent.
    
    Args:
        layer_map: Mapping of latent_idx (as string) to list of context entries
        window_size: Size of window around max-activating token
        
    Returns:
        Dictionary mapping latent_idx to list of token lists (one per context).
        Each inner list contains the tokens in the window around maxValueTokenIndex.
    """
    latent_window_tokens: Dict[int, List[List[Any]]] = {}
    
    for latent_str, contexts in layer_map.items():
        try:
            latent_idx = int(latent_str)
        except (ValueError, TypeError):
            continue
            
        if not isinstance(contexts, list) or not contexts:
            continue
        if context_size is not None and context_size > 0:
            contexts = contexts[:context_size]
        
        window_token_lists: List[List[Any]] = []
        
        for entry in contexts:
            tokens = entry.get("tokens") if isinstance(entry, dict) else None
            if not isinstance(tokens, list) or not tokens:
                continue

            max_idx = None
            if isinstance(entry, dict):
                mi = entry.get("maxValueTokenIndex")
                if isinstance(mi, int):
                    max_idx = mi
                elif isinstance(entry.get("values"), list) and entry.get("values"):
                    # Fallback: compute argmax over values if provided
                    try:
                        values_list = entry.get("values")
                        max_idx = max(range(len(values_list)), key=lambda i: values_list[i])
                    except Exception:
                        max_idx = None

            if isinstance(max_idx, int):
                start = max(0, max_idx - window_size)
                end = min(len(tokens), max_idx + window_size + 1)
                sub_tokens = tokens[start:end]
                window_token_lists.append(sub_tokens)
            else:
                # If no max index info, use full context
                window_token_lists.append(tokens)
        
        if window_token_lists:
            latent_window_tokens[latent_idx] = window_token_lists
    
    return latent_window_tokens

In [ ]:
def _tokens_to_text(tokens: Sequence[str]) -> str:
    return "".join(token.replace("▁", " ").replace("<0x0A>", "\n") for token in tokens)


def _has_boundary_match(hay: str, needle: str) -> bool:
    if not needle:
        return False
    if any(ch.isalnum() for ch in needle):
        idx = 0
        hay_len = len(hay)
        needle_len = len(needle)
        while True:
            pos = hay.find(needle, idx)
            if pos == -1:
                return False
            left_ok = pos == 0 or not hay[pos - 1].isalnum()
            right_ok = (pos + needle_len) >= hay_len or not hay[pos + needle_len].isalnum()
            if left_ok and right_ok:
                return True
            idx = pos + 1
    return needle in hay

In [ ]:
def get_latent_to_matching_tokens(
    latent_window_tokens: Dict[int, List[List[Any]]],
    min_context_matches: int = 1,
) -> Dict[int, List[str]]:
    """Get mapping from latent_idx to list of tokens that appear in minimum contexts.
    
    Args:
        latent_window_tokens: Dict mapping latent_idx to list of token lists
        min_context_matches: Minimum number of contexts that must contain the token
        
    Returns:
        Dictionary mapping latent_idx to list of token strings that meet the threshold
    """
    latent_to_tokens: Dict[int, List[str]] = {}
    
    for latent_idx, token_lists in latent_window_tokens.items():
        # Collect unique tokens from THIS latent's contexts only
        unique_tokens: Set[str] = set()
        for tokens in token_lists:
            for token in tokens:
                if isinstance(token, str):
                    unique_tokens.add(token)
        
        # Convert raw tokens to searchable labels
        token_labels: Set[str] = set()
        for token in unique_tokens:
            label = token.replace("▁", " ").replace("<0x0A>", "\n").strip()
            if label:
                token_labels.add(label)
        
        # Convert token lists to window texts
        window_texts: List[str] = []
        for tokens in token_lists:
            text = _tokens_to_text(tokens)
            if text:
                normalized = " ".join(text.replace("\r", " ").replace("\n", " ").split())
                window_texts.append(normalized)
        
        if not window_texts:
            continue
        
        # Check which tokens have enough matches in THIS latent
        matching_tokens: List[str] = []
        for label in token_labels:
            match_count = sum(1 for text in window_texts if _has_boundary_match(text, label))
            if match_count >= max(1, int(min_context_matches)):
                matching_tokens.append(label)
        
        if matching_tokens:
            latent_to_tokens[latent_idx] = matching_tokens
    
    return latent_to_tokens

In [ ]:
import os
output_dir = "cache_dir/"

os.makedirs(output_dir, exist_ok=True)

for i in range(26):
    layer_map = _load_saved_layer_contexts(
        saved_dir="./",
        layer_index=i,
        top_k=20
    )
    latent_window_map = extract_latent_window_tokens(
        layer_map=layer_map,
        window_size=5,
        context_size=20
    )
    latent_window_token_map = get_latent_to_matching_tokens(
        latent_window_tokens=latent_window_map,
        min_context_matches=15
    )

    with open(output_dir + f"L{i}.top20.match.json", "w") as f:
        json.dump(latent_window_token_map, f, indent = 2)